## load_silver_fred
Conforms `bronze.fred_series_observations` into `silver.fact_fred_series` — **long, native cadence** (one row per `series_id`, `observation_date`). National macro series; **no geo, no resample, no pivot** (cadence reconciliation is a Gold concern). `series_id` joins `dim_fred_series`.

**Transforms:** cast `observation_date`→DATE, `value`→DOUBLE (kept raw — dollar series keep cents; Gold rounds when serving). The FRED missing sentinel `'.'` and any malformed value/date → `silver.quarantine` (`null_value` / `cast_failed:<col>`). MERGE on `(series_id, observation_date)`.

In [ ]:
%run "../libs/notebook_init"

In [ ]:
STEP_SEQUENCE = 1
SOURCE_SYSTEM = "fred"
SOURCE_TABLE  = f"{BRONZE}.fred_series_observations"
TARGET_TABLE  = f"{SILVER}.fact_fred_series"
QUARANTINE    = f"{SILVER}.quarantine"

In [ ]:
# Open the pipeline_step_log row (RUNNING).
nb = Utils.get_notebook_context(dbutils)
step = StepLog(
    spark, AUDIT, dbutils,
    pipeline_run_id = PIPELINE_RUN_ID,
    step_sequence   = STEP_SEQUENCE,
    notebook_folder = nb["notebook_folder"],
    notebook_name   = nb["notebook_name"],
    layer           = "silver",
    target_table    = TARGET_TABLE,
)
print(f"load_silver_fred: step_log_id={step.step_log_id}")

In [ ]:
# Read + type. reject_reason marks bad rows: unparseable date, the '.' missing sentinel, or a
# value that is non-empty but won't cast. value kept raw DOUBLE (no rounding at Silver).
try:
    bronze = spark.table(SOURCE_TABLE)
    rows_read = bronze.count()

    obs_date = F.to_date(F.col("observation_date"))
    reject_reason = (
        F.when(obs_date.isNull(), F.lit("cast_failed:observation_date"))
         .when(F.col("value") == F.lit("."), F.lit("null_value"))
         .when((F.col("value").isNotNull()) & (F.trim(F.col("value")) != F.lit(""))
               & (F.col("value").cast("double").isNull()), F.lit("cast_failed:value"))
         .otherwise(F.lit(None))
    )
    staged = bronze.select(
        F.col("series_id"),
        obs_date.alias("observation_date"),
        F.col("value").cast("double").alias("value"),
        F.col("source_file_path"),
        reject_reason.alias("reject_reason"),
        F.to_json(F.struct(*[F.col(col_name) for col_name in bronze.columns])).alias("raw_payload"),
    )
    step.rows_read = rows_read
    print(f"load_silver_fred: read {rows_read:,} observations")
except Exception as e:
    step.fail(e); raise

In [ ]:
# Quarantine rejects (idempotent), MERGE good rows on (series_id, observation_date), audit.
transform_started = datetime.now(timezone.utc)
rows_rejected = rows_inserted = rows_updated = 0
try:
    good = staged.where(F.col("reject_reason").isNull())
    bad  = staged.where(F.col("reject_reason").isNotNull())
    rows_rejected = bad.count()

    spark.sql(f"DELETE FROM {QUARANTINE} WHERE source_system = '{SOURCE_SYSTEM}'")
    if rows_rejected > 0:
        bad.select(
            F.expr("uuid()").alias("quarantine_id"),
            F.lit(SOURCE_SYSTEM).alias("source_system"),
            F.col("source_file_path"),
            F.concat_ws("|", F.col("series_id"), F.col("observation_date").cast("string")).alias("natural_key"),
            F.col("raw_payload"),
            F.col("reject_reason").alias("quarantine_reason"),
            F.current_timestamp().alias("quarantined_ts"),
        ).write.format("delta").mode("append").saveAsTable(QUARANTINE)

    good.select(
        F.col("series_id"), F.col("observation_date"), F.col("value"),
        F.current_timestamp().alias("inserted_ts"),
        F.current_timestamp().alias("updated_ts"),
    ).createOrReplaceTempView("fred_fact_staging")

    metrics = spark.sql(f"""
        MERGE INTO {TARGET_TABLE} t USING fred_fact_staging s
        ON t.series_id = s.series_id AND t.observation_date = s.observation_date
        WHEN MATCHED THEN UPDATE SET t.value=s.value, t.updated_ts=s.updated_ts
        WHEN NOT MATCHED THEN INSERT (series_id, observation_date, value, inserted_ts, updated_ts)
            VALUES (s.series_id, s.observation_date, s.value, s.inserted_ts, s.updated_ts)
    """).first().asDict()
    rows_inserted = metrics.get("num_inserted_rows") or 0
    rows_updated  = metrics.get("num_updated_rows") or 0

    step.rows_written = rows_inserted
    transform_detail_log_insert(
        spark, AUDIT, PIPELINE_RUN_ID, step.step_log_id, SOURCE_TABLE, TARGET_TABLE,
        status=STATUS_SUCCEEDED, started_timestamp=transform_started, rows_read=step.rows_read,
        rows_written=rows_inserted, rows_inserted=rows_inserted, rows_updated=rows_updated,
        rows_rejected=rows_rejected, ended_timestamp=datetime.now(timezone.utc))
    step.succeed()
    print(f"load_silver_fred: inserted={rows_inserted:,} updated={rows_updated:,} "
          f"quarantined={rows_rejected:,} (read={step.rows_read:,})")
except Exception as e:
    transform_detail_log_insert(
        spark, AUDIT, PIPELINE_RUN_ID, step.step_log_id, SOURCE_TABLE, TARGET_TABLE,
        status=STATUS_FAILED, started_timestamp=transform_started, rows_read=step.rows_read,
        rows_rejected=rows_rejected, error_message=f"{type(e).__name__}: {e}",
        ended_timestamp=datetime.now(timezone.utc))
    step.fail(e); raise